# TrustDoc — The Trust Layer (temperature scaling, ECE, reliability, flagging)

This is the project's actual differentiator: calibrating the classifier's confidence so low-confidence predictions can be routed to human review instead of shipped as fact.

Loads the already-trained classifier from the HF Hub (`vxa8502/trustdoc-classifier`, public, no auth needed) and runs inference on `rvl_cdip_mini`'s **validation** split (to fit temperature) and **test** split (held out, to report ECE/reliability/business metrics honestly -- fitting and evaluating on the same data would be methodologically wrong).

The calibration math (`fit_temperature`, `compute_ece`/`compute_mce`, `plot_reliability_diagram`, `threshold_report`) mirrors `src/calibrate/*.py` in the repo, which has real unit tests (`tests/test_calibrate.py`) -- including a check that temperature scaling provably preserves accuracy (verified against a synthetic overconfident example before ever touching real data). It's inlined here rather than imported since Kaggle notebooks don't have access to this repo's files.

**Lessons applied here:** detect dataset schema instead of assuming it; box-clip to [0,1000] defensively; use "Save Version -> Save & Run All (Commit)" for the real run.

In [ ]:
import torch, subprocess
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"], capture_output=True, text=True).stdout)
print("torch.cuda.is_available():", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("No GPU visible — on Kaggle: Settings > Accelerator > GPU T4x2/P100. On Colab: Runtime > Change runtime type > GPU.")

In [ ]:
import importlib, subprocess, sys

def ensure(pkg, import_name=None):
    import_name = import_name or pkg
    try:
        importlib.import_module(import_name)
        print(f"{pkg}: already available, skipping install")
    except ImportError:
        print(f"{pkg}: not found, installing with --no-deps")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", pkg], check=True)

ensure("datasets")

import transformers
print("transformers version (as provided by this environment):", transformers.__version__)

In [ ]:
from transformers import AutoProcessor, AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained("vxa8502/trustdoc-classifier").to("cuda")
processor = AutoProcessor.from_pretrained("vxa8502/trustdoc-classifier", apply_ocr=False)
model.eval()
# Pinned into calibration_summary.json below so src/pipeline.py can catch a future retrain
# silently drifting from this fitted temperature.
model_revision = model.config._commit_hash
print("loaded classifier: num_labels =", model.config.num_labels, "revision =", model_revision)

In [ ]:
from datasets import load_dataset

dataset = load_dataset("dvgodoy/rvl_cdip_mini")
print("splits:", list(dataset.keys()))

image_column = next(c for c in ("image", "img", "pixel_values") if c in dataset["train"].column_names)
label_column = next(c for c in ("label", "labels") if c in dataset["train"].column_names)
print("using image column:", image_column, "| label column:", label_column)

val_split = next((s for s in ("validation", "valid", "val") if s in dataset), None)
test_split = "test" if "test" in dataset else None
assert val_split and test_split, "need both a validation split (to fit T) and a test split (to report on, held out from fitting)"
print(f"val: {len(dataset[val_split])} examples, test: {len(dataset[test_split])} examples")

# Same box-safety check as the classifier training notebook: detect whether word_boxes need
# normalizing to LayoutLMv3's required 0-1000 scale, and always clip defensively regardless.
_all_boxes = dataset[val_split]["word_boxes"]
_flat_coords = [c for boxes in _all_boxes for b in boxes for c in b]
_max_coord = max(_flat_coords) if _flat_coords else 0
needs_normalization = _max_coord > 1000
print("word_boxes need normalization:", needs_normalization, f"(max coord seen: {_max_coord})")

def normalize_box(box, width, height):
    x0, y0, x1, y1 = box
    return [int(1000 * x0 / width), int(1000 * y0 / height), int(1000 * x1 / width), int(1000 * y1 / height)]

def clip_box(box):
    return [max(0, min(1000, c)) for c in box]

def prepare(examples):
    images = [img.convert("RGB") for img in examples[image_column]]
    words = examples["ocr_words"]
    if needs_normalization:
        boxes = [
            [normalize_box(b, w, h) for b in ex_boxes]
            for ex_boxes, w, h in zip(examples["word_boxes"], examples["width"], examples["height"])
        ]
    else:
        boxes = examples["word_boxes"]
    boxes = [[clip_box(b) for b in ex_boxes] for ex_boxes in boxes]
    encoding = processor(images, words, boxes=boxes, truncation=True, padding="max_length")
    encoding["labels"] = examples[label_column]
    return encoding

columns_to_remove = [c for c in dataset[val_split].column_names if c != label_column]
val_ds = dataset[val_split].map(prepare, batched=True, batch_size=8, remove_columns=columns_to_remove)
val_ds.set_format("torch")
test_ds = dataset[test_split].map(prepare, batched=True, batch_size=8, remove_columns=columns_to_remove)
test_ds.set_format("torch")

In [ ]:
from torch.utils.data import DataLoader

@torch.no_grad()
def get_logits_and_labels(torch_dataset, batch_size=16):
    loader = DataLoader(torch_dataset, batch_size=batch_size)
    all_logits, all_labels = [], []
    for batch in loader:
        labels = batch.pop("labels")
        batch = {k: v.to("cuda") for k, v in batch.items()}
        outputs = model(**batch)
        all_logits.append(outputs.logits.cpu())
        all_labels.append(labels)
    return torch.cat(all_logits), torch.cat(all_labels)

val_logits, val_labels = get_logits_and_labels(val_ds)
test_logits, test_labels = get_logits_and_labels(test_ds)
print("val logits:", val_logits.shape, "test logits:", test_logits.shape)

val_acc = (val_logits.argmax(dim=1) == val_labels).float().mean().item()
test_acc = (test_logits.argmax(dim=1) == test_labels).float().mean().item()
print(f"val accuracy: {val_acc:.3f}, test accuracy: {test_acc:.3f}")

In [ ]:
# --- calibration math, mirrors src/calibrate/*.py (unit-tested there; see tests/test_calibrate.py) ---
import numpy as np
import torch.nn.functional as F

def fit_temperature(logits, labels, max_iter=50, lr=0.01):
    logits = logits.detach()
    labels = labels.detach()
    log_T = torch.zeros(1, requires_grad=True)
    optimizer = torch.optim.LBFGS([log_T], lr=lr, max_iter=max_iter)

    def closure():
        optimizer.zero_grad()
        T = log_T.exp()
        loss = F.cross_entropy(logits / T, labels)
        loss.backward()
        return loss

    optimizer.step(closure)
    return log_T.exp().item()

def confidence_and_correctness(probs, labels):
    predictions = probs.argmax(axis=1)
    confidences = probs.max(axis=1)
    correct = (predictions == labels).astype(float)
    return confidences, correct

def binned_stats(confidences, correct, n_bins):
    # (lo, hi, proportion, accuracy, mean_confidence, count) per bin. `count` is exact (not
    # derived from the rounded proportion) so sparsely populated bins can be flagged honestly.
    bin_boundaries = np.linspace(0, 1, n_bins + 1)
    stats = []
    for lo, hi in zip(bin_boundaries[:-1], bin_boundaries[1:]):
        in_bin = (confidences > lo) & (confidences <= hi)
        count_in_bin = int(in_bin.sum())
        prop_in_bin = in_bin.mean()
        if count_in_bin > 0:
            acc_in_bin = correct[in_bin].mean()
            conf_in_bin = confidences[in_bin].mean()
        else:
            acc_in_bin = conf_in_bin = 0.0
        stats.append((lo, hi, prop_in_bin, acc_in_bin, conf_in_bin, count_in_bin))
    return stats

def compute_ece(confidences, correct, n_bins=15, stats=None):
    # stats: pass a precomputed binned_stats(...) result to avoid recomputing it -- compute_mce
    # and plot_reliability_diagram below consume the exact same per-bin stats.
    if stats is None:
        stats = binned_stats(confidences, correct, n_bins)
    return sum(abs(a - c) * p for _, _, p, a, c, _ in stats)

def compute_mce(confidences, correct, n_bins=15, stats=None):
    if stats is None:
        stats = binned_stats(confidences, correct, n_bins)
    gaps = [abs(a - c) for _, _, p, a, c, _ in stats if p > 0]
    return max(gaps) if gaps else 0.0

def plot_reliability_diagram(confidences, correct, n_bins=15, title="Reliability Diagram", ax=None, count_ax=None, stats=None):
    import matplotlib.pyplot as plt
    if stats is None:
        stats = binned_stats(confidences, correct, n_bins)
    bin_boundaries = np.linspace(0, 1, n_bins + 1)
    bin_centers = (bin_boundaries[:-1] + bin_boundaries[1:]) / 2
    accs = [s[3] for s in stats]
    counts = [s[5] for s in stats]
    width = (1.0 / n_bins) * 0.9
    if ax is None:
        ax = plt.subplots(figsize=(5, 5))[1]
    ax.bar(bin_centers, accs, width=width, edgecolor="black", alpha=0.7, label="Accuracy")
    ax.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Perfect calibration")
    ax.set_ylabel("Accuracy"); ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.set_title(title); ax.legend()
    if count_ax is not None:
        count_ax.bar(bin_centers, counts, width=width, color="gray", alpha=0.7)
        count_ax.set_xlabel("Confidence"); count_ax.set_ylabel("Count")
        count_ax.set_xlim(0, 1)
    else:
        ax.set_xlabel("Confidence")
    return ax

def threshold_report(confidences, correct, thresholds=None):
    if thresholds is None:
        thresholds = np.linspace(0.5, 0.99, 50)
    rows = []
    for t in thresholds:
        mask = confidences >= t
        n_auto = int(mask.sum())
        rows.append({
            "threshold": float(t),
            "auto_approve_rate": float(mask.mean()),
            "precision": float(correct[mask].mean()) if n_auto > 0 else None,
            "n_auto_approved": n_auto,
        })
    return rows


In [ ]:
# Fit T on the validation split (never touched again), report everything else on the held-out test split.
T = fit_temperature(val_logits, val_labels)
print(f"fitted temperature T = {T:.3f}")

probs_before = F.softmax(test_logits, dim=1).numpy()
probs_after = F.softmax(test_logits / T, dim=1).numpy()
labels_np = test_labels.numpy()

conf_before, correct_before = confidence_and_correctness(probs_before, labels_np)
conf_after, correct_after = confidence_and_correctness(probs_after, labels_np)

# Sanity check on the notebook's own math, not just the unit tests: scaling logits by a positive
# constant must not change predictions -- if this fails, something upstream is wrong.
assert np.array_equal(correct_before, correct_after), "temperature scaling changed predictions -- bug"

# Compute binned_stats once per split and share it across ECE/MCE/the reliability diagram
# (cell below) instead of each of those three recomputing an identical bin scan.
stats_before = binned_stats(conf_before, correct_before, n_bins=15)
stats_after = binned_stats(conf_after, correct_after, n_bins=15)
ece_before, mce_before = compute_ece(conf_before, correct_before, stats=stats_before), compute_mce(conf_before, correct_before, stats=stats_before)
ece_after, mce_after = compute_ece(conf_after, correct_after, stats=stats_after), compute_mce(conf_after, correct_after, stats=stats_after)
print(f"BEFORE calibration: ECE={ece_before:.4f} MCE={mce_before:.4f}")
print(f"AFTER  calibration: ECE={ece_after:.4f} MCE={mce_after:.4f}")

In [ ]:
import os
os.makedirs("results", exist_ok=True)  # not auto-created here, unlike the training notebooks (Trainer does that internally via output_dir)

import matplotlib.pyplot as plt

# 2x2: accuracy bars on top, per-bin sample counts below -- without the count row, a sparsely
# populated bin can show a misleadingly "perfect" accuracy bar that's really just noise.
fig, axes = plt.subplots(2, 2, figsize=(11, 6), gridspec_kw={"height_ratios": [3, 1]}, sharex=True)
plot_reliability_diagram(conf_before, correct_before, title=f"Before (ECE={ece_before:.3f})", ax=axes[0, 0], count_ax=axes[1, 0], stats=stats_before)
plot_reliability_diagram(conf_after, correct_after, title=f"After, T={T:.2f} (ECE={ece_after:.3f})", ax=axes[0, 1], count_ax=axes[1, 1], stats=stats_after)
plt.tight_layout()
plt.savefig("results/reliability_diagrams.png", dpi=150)
plt.show()


In [ ]:
import json

report = threshold_report(conf_after, correct_after)

thresholds = [r["threshold"] for r in report]
auto_rates = [r["auto_approve_rate"] for r in report]
precisions = [r["precision"] for r in report]

fig, ax1 = plt.subplots(figsize=(7, 5))
ax1.plot(thresholds, auto_rates, label="Auto-approve rate", color="tab:blue")
ax1.set_xlabel("Confidence threshold")
ax1.set_ylabel("Auto-approve rate", color="tab:blue")
ax2 = ax1.twinx()
ax2.plot(thresholds, precisions, label="Precision (auto-approved)", color="tab:red")
ax2.set_ylabel("Precision among auto-approved", color="tab:red")
plt.title("Human-in-the-loop trade-off: auto-approve rate vs. precision")
plt.tight_layout()
plt.savefig("results/threshold_tradeoff.png", dpi=150)
plt.show()

# business framing at a couple of concrete thresholds
for t in (0.7, 0.9):
    row = min(report, key=lambda r: abs(r["threshold"] - t))
    print(f"At confidence >= {row['threshold']:.2f}: {row['auto_approve_rate']*100:.1f}% of documents "
          f"auto-approved with {row['precision']*100:.1f}% precision among them "
          f"({row['n_auto_approved']}/{len(conf_after)} documents).")

with open("results/calibration_summary.json", "w") as f:
    json.dump({
        "model_revision": model_revision,
        "temperature": T,
        "val_accuracy": val_acc,
        "test_accuracy": test_acc,
        "ece_before": ece_before, "mce_before": mce_before,
        "ece_after": ece_after, "mce_after": mce_after,
        "threshold_report": report,
    }, f, indent=2)
print("saved results/calibration_summary.json")

## Checklist
- [ ] val/test accuracy printed above roughly match the current classifier's validation accuracy (see results/classifier_report.json's "accuracy" field for the latest number -- confirms the model loaded and preprocessing pipeline are consistent with training)
- [ ] `correct_before == correct_after` assertion passed (temperature scaling did not change predictions)
- [ ] ECE/MCE after calibration are lower than before — MCE can go the other way on small/sparse bins even when ECE improves; check the count panel before treating an MCE increase as a problem
- [ ] Reliability diagrams (with per-bin count panels) saved to `results/reliability_diagrams.png` — check whether any tall/short accuracy bar corresponds to a tiny count before reading too much into it
- [ ] Threshold trade-off chart saved to `results/threshold_tradeoff.png`, `results/calibration_summary.json` saved
- [ ] Ran via "Save Version -> Save & Run All (Commit)", not the live interactive session
- [ ] Record T, ECE before/after, and the business-framed threshold examples
